# Step 2: Clean the cells and confirm the dopamine neurons

This notebook starts from the zip file saved in your Google Drive in step 1, so it works in a fresh Colab session.

**How to run:** click each grey box from top to bottom and press **Shift + Enter**, waiting for the green tick before the next one. Stop at Cells 4 and 7 and send Claude a screenshot.

**Total time:** about 45–75 minutes. Keep the tab open and your Mac awake.

## Cell 1 — Connect Google Drive and unzip the step 1 result (3–5 minutes)
A pop-up asks to connect Google Drive; approve it. The file list at the end should show matrix.mtx.gz (about 953 MB).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp /content/drive/MyDrive/GSE178265_DA_extract.zip /content/
!cd /content && unzip -o -q GSE178265_DA_extract.zip
!ls -lh /content/GSE178265_DA_extract

## Cell 2 — Install the single-cell tools (1–2 minutes)
Lots of text may scroll by. That is normal.

In [ ]:
!pip install -q scanpy harmonypy igraph leidenalg

## Cell 3 — Load the 50,058 cells (10–20 minutes)
**Check:** it should say about 50,058 cells x 41,625 genes, then 30,058 DA_candidate and 20,000 reference_sample.

In [ ]:
import gzip, time
import numpy as np, pandas as pd, scanpy as sc, anndata as ad
from scipy import sparse

IN = "/content/GSE178265_DA_extract"
t0 = time.time()
with gzip.open(f"{IN}/matrix.mtx.gz", "rt") as f:
    skip = 0
    for line in f:
        skip += 1
        if not line.startswith("%"):
            n_genes, n_cells, n_vals = (int(x) for x in line.split())
            break

rows, cols, vals = [], [], []
for ch in pd.read_csv(f"{IN}/matrix.mtx.gz", sep=" ", header=None, skiprows=skip,
                      chunksize=20_000_000, names=["g", "c", "v"],
                      dtype={"g": np.int32, "c": np.int32, "v": np.float32}):
    rows.append(ch["c"].to_numpy() - 1)   # cells become rows
    cols.append(ch["g"].to_numpy() - 1)   # genes become columns
    vals.append(ch["v"].to_numpy())
rows = np.concatenate(rows)   # joined one at a time to keep memory low
cols = np.concatenate(cols)
vals = np.concatenate(vals)
X = sparse.csr_matrix((vals, (rows, cols)), shape=(n_cells, n_genes), dtype=np.float32)
del rows, cols, vals

genes = pd.read_csv(f"{IN}/features.tsv.gz", sep="\t", header=None)[0].astype(str).to_numpy()
info = pd.read_csv(f"{IN}/cell_info.csv")
adata = ad.AnnData(X=X, obs=info.set_index("barcode"), var=pd.DataFrame(index=genes))
adata.var_names_make_unique()
adata.obs["sample"] = adata.obs_names.str.rsplit("_", n=1).str[0]
print(f"Loaded {adata.n_obs:,} cells x {adata.n_vars:,} genes in {(time.time() - t0) / 60:.1f} min")
print(adata.obs["group"].value_counts())

## Cell 4 — Quality numbers and the list of samples (1–2 minutes)
**Send Claude a screenshot of the table at the end.** It lists every donor sample; Claude uses it to match each sample to Parkinson's or control.

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
print(adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe().round(1))
samples = adata.obs.groupby("sample", observed=True)["group"].value_counts().unstack(fill_value=0)
samples.to_csv("/content/drive/MyDrive/GSE178265_samples.csv")
pd.set_option("display.max_rows", 200)
print(f"\n{samples.shape[0]} samples. Full list saved to My Drive as GSE178265_samples.csv")
samples

## Cell 4b — Match samples to GEO records (added 23 Oct)

In [ ]:
# CELL 4b — Match each sample to Parkinson's or control using GEO's own records (1–2 minutes)
import gzip, re, urllib.request
base = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE178nnn/GSE178265/matrix/"
listing = urllib.request.urlopen(base).read().decode()
files = sorted(set(re.findall(r'href="(GSE178265[^"]*series_matrix\.txt\.gz)"', listing)))
print("GEO record files:", files)

recs = []
for fn in files:
    urllib.request.urlretrieve(base + fn, f"/content/{fn}")
    fields = {}
    with gzip.open(f"/content/{fn}", "rt", errors="replace") as fh:
        for line in fh:
            if line.startswith("!series_matrix_table_begin"):
                break
            if line.startswith("!Sample_"):
                parts = line.rstrip("\n").split("\t")
                key = parts[0][8:]
                vals = [p.strip('"') for p in parts[1:]]
                k, n = key, 2
                while k in fields:
                    k = f"{key}_{n}"; n += 1
                fields[k] = vals
    n_s = len(fields.get("title", []))
    for i in range(n_s):
        r = {"file": fn}
        for k, v in fields.items():
            if i < len(v) and (k in ("title", "geo_accession", "organism_ch1", "source_name_ch1") or k.startswith("characteristics")):
                val = v[i]
                if k.startswith("characteristics") and ":" in val:
                    name, val = val.split(":", 1)
                    r[name.strip().lower()] = val.strip()
                else:
                    r[k] = val
        recs.append(r)
geo = pd.DataFrame(recs)
geo = geo[geo.get("organism_ch1", pd.Series("Homo sapiens", index=geo.index)).astype(str).str.contains("sapiens")]
print(f"{len(geo)} human samples in GEO records\n")

def norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())
geo_keys = {norm(t): i for i, t in zip(geo.index, geo["title"])}
match = {}
for s in samples.index:
    ns = norm(s)
    hit = geo_keys.get(ns)
    if hit is None:
        cands = [i for k, i in geo_keys.items() if k and (k in ns or ns in k)]
        hit = cands[0] if len(cands) == 1 else None
    match[s] = hit
merged = samples.copy()
merged["geo_row"] = pd.Series(match)
for col in geo.columns:
    if col not in ("file", "organism_ch1"):
        merged[col] = merged["geo_row"].map(geo[col])
merged = merged.drop(columns="geo_row")
merged.to_csv("/content/drive/MyDrive/GSE178265_samples_with_GEO.csv")

print(f"Matched {merged['title'].notna().sum()} of {len(merged)} samples to a GEO record")
info_cols = [c for c in merged.columns if c not in ("DA_candidate", "reference_sample", "title", "geo_accession")]
for c in info_cols:
    if merged[c].nunique(dropna=True) <= 12:
        print(f"\n== {c} ==")
        print(merged.groupby(merged[c].fillna("(no match)"))[["DA_candidate", "reference_sample"]].agg(["count", "sum"]).to_string())
    else:
        print(f"\n== {c} == ({merged[c].nunique()} different values, e.g. {list(merged[c].dropna().unique()[:4])})")
unmatched = merged.index[merged["title"].isna()].tolist()
print(f"\nUnmatched samples ({len(unmatched)}): {unmatched[:15]}")

## Cell 4c — Attach donor details, remove caudate (added 23 Oct)

In [ ]:
# CELL 4c — Attach donor details, remove caudate samples (seconds)
meta = merged[["tissue", "source_name_ch1", "status", "disease", "sex", "age", "pmi",
               "facs classification", "geo_accession"]].rename(columns={"facs classification": "facs"})
adata.obs = adata.obs.join(meta, on="sample")
caudate = (adata.obs["tissue"].astype(str).str.contains("audate") |
           adata.obs["source_name_ch1"].astype(str).str.contains("audate"))
print(f"Removing {int(caudate.sum()):,} caudate cells")
adata = adata[~caudate.to_numpy()].copy()
adata.obs["donor"] = adata.obs["sample"].astype(str).str.extract(r"(\d{4})", expand=False)

donors = adata.obs.groupby("donor", observed=True).agg(
    status=("status", "first"), sex=("sex", "first"), age=("age", "first"),
    samples=("sample", "nunique"), DA_candidates=("group", lambda g: int((g == "DA_candidate").sum())))
print(f"\n{adata.n_obs:,} cells from {donors.shape[0]} donors\n")
print(donors.groupby("status")[["DA_candidates"]].agg(["count", "sum"]).rename(columns={"count": "donors", "sum": "DA cells"}))
print("\nEach donor should have ONE status, sex and age. Check for conflicts:")
chk = adata.obs.groupby("donor", observed=True)[["status", "sex", "age"]].nunique()
print("None found" if (chk.max(axis=1) <= 1).all() else chk[chk.max(axis=1) > 1])
donors.sort_values(["status", "age"])

## Cell 5 — Remove low-quality cells (seconds)
Keeps cells with at least 500 genes detected and under 5% mitochondrial reads.

In [ ]:
before = adata.n_obs
adata = adata[(adata.obs["n_genes_by_counts"] >= 500) & (adata.obs["pct_counts_mt"] < 5)].copy()
sc.pp.filter_genes(adata, min_cells=10)
print(f"Kept {adata.n_obs:,} of {before:,} cells and {adata.n_vars:,} genes")
print(adata.obs["group"].value_counts())

## Cell 6 — Group similar cells into clusters across donors (15–30 minutes)
This is the slowest cell. It ends with a line like '18 clusters found'.

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=3000, batch_key=None)
sc.tl.pca(adata, n_comps=30, mask_var="highly_variable")
import harmonypy
ho = harmonypy.run_harmony(adata.obsm["X_pca"], adata.obs, "sample", max_iter_harmony=20)
Z = np.asarray(ho.Z_corr)
adata.obsm["X_pca_harmony"] = Z if Z.shape[0] == adata.n_obs else Z.T
sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=20)
sc.tl.leiden(adata, resolution=0.5, flavor="igraph", n_iterations=2, directed=False)
sc.tl.umap(adata)
print(f"{adata.obs['leiden'].nunique()} clusters found")

## Cell 7 — Which clusters are dopamine neurons? (seconds)
**Send Claude a screenshot of the table and the four pictures.** Clusters where most cells make TH plus SLC6A3 or SLC18A2 are marked is_DA = True. Claude will check the call before you save.

In [ ]:
counts = adata.layers["counts"]
def frac_expr(g):
    if g not in adata.var_names:
        print(f"Note: {g} not found after filtering")
        return np.zeros(adata.n_obs, dtype=bool)
    j = adata.var_names.get_loc(g)
    return (counts[:, j].toarray().ravel() > 0)
th = frac_expr("TH")
dat_or_vmat = frac_expr("SLC6A3") | frac_expr("SLC18A2")
table = pd.DataFrame({"cluster": adata.obs["leiden"].to_numpy(), "TH": th, "SLC6A3_or_SLC18A2": dat_or_vmat})
table = table.groupby("cluster").agg(cells=("TH", "size"), TH=("TH", "mean"), SLC6A3_or_SLC18A2=("SLC6A3_or_SLC18A2", "mean"))
table["is_DA"] = (table["TH"] >= 0.5) & (table["SLC6A3_or_SLC18A2"] >= 0.3)
print(table.round(2).sort_values("TH", ascending=False).to_string())
print(f"\nDopamine neurons: {table.loc[table['is_DA'], 'cells'].sum():,} cells in {table['is_DA'].sum()} clusters")
sc.pl.umap(adata, color=["leiden", "TH", "SLC6A3", "group"], ncols=2, legend_loc="on data", frameon=False)

## Cell 7b — Confirm DA clusters by marker strength (added 23 Oct; replaces the circular Cell 7 table)

In [ ]:
# CELL 7b — Check each cluster by how STRONGLY it makes each cell type's marker genes (seconds)
markers = {
    "DA_neuron": ["TH", "SLC6A3", "SLC18A2", "DDC", "KCNJ6", "RET"],
    "GABA_neuron": ["GAD1", "GAD2"],
    "Glut_neuron": ["SLC17A6", "SLC17A7"],
    "Oligodendrocyte": ["MBP", "PLP1", "MOG"],
    "OPC": ["PDGFRA", "VCAN"],
    "Astrocyte": ["AQP4", "GFAP", "SLC1A2"],
    "Microglia": ["P2RY12", "CX3CR1", "CSF1R"],
    "Endothelial": ["CLDN5", "FLT1"],
}
present = {t: [g for g in gs if g in adata.var_names] for t, gs in markers.items()}
all_genes = sorted({g for gs in present.values() for g in gs} |
                   {g for g in ["SOX6", "AGTR1", "CALB1"] if g in adata.var_names})
expr = sc.get.obs_df(adata, keys=all_genes)          # log-normalised expression
cl = adata.obs["leiden"].astype(str).to_numpy()

score = pd.DataFrame({t: expr[gs].mean(axis=1) for t, gs in present.items() if gs})
by_cl = score.groupby(cl).mean()
is_cand = pd.Series((adata.obs["group"].to_numpy() == "DA_candidate").astype(float))
summary = pd.DataFrame({
    "cells": pd.Series(cl).value_counts(),
    "pct_candidates": is_cand.groupby(cl).mean().mul(100).round(0),
})
summary = summary.join(by_cl.round(2))
summary["best_guess"] = by_cl.idxmax(axis=1)
summary["DA_margin"] = (by_cl["DA_neuron"] - by_cl.drop(columns="DA_neuron").max(axis=1)).round(2)
for g in ["SOX6", "AGTR1", "CALB1"]:
    if g in expr:
        summary[g] = expr[g].groupby(cl).mean().round(2)
summary.index.name = "cluster"
summary = summary.sort_values("DA_neuron", ascending=False)

suggest = summary.index[(summary["best_guess"] == "DA_neuron") & (summary["DA_margin"] >= 0.3)]
table = table.copy()
table.index = table.index.astype(str)
table["is_DA"] = table.index.isin(suggest)

pd.set_option("display.width", 250)
print(summary.to_string())
print(f"\nSuggested dopamine-neuron clusters: {sorted(suggest, key=int)}")
print(f"Cells in them: {int(summary.loc[suggest, 'cells'].sum()):,}")
sc.pl.dotplot(adata, var_names=present | {"DA_subtype": [g for g in ["SOX6", "AGTR1", "CALB1"] if g in adata.var_names]},
              groupby="leiden", standard_scale="var")

## Cell 8 — Save the dopamine neurons to Google Drive (5–10 minutes)
Only run this after Claude has checked Cell 7. The result, GSE178265_DA_neurons.h5ad, lands in My Drive.

In [ ]:
da_clusters = table.index[table["is_DA"]].astype(str)
da = adata[adata.obs["leiden"].astype(str).isin(da_clusters)].copy()
da.X = da.layers["counts"]
del da.layers["counts"]
for k in ["X_pca", "X_pca_harmony"]:
    da.obsm.pop(k, None)
sc.pp.filter_genes(da, min_cells=10)
da.obs["DA_cluster"] = da.obs.pop("leiden").astype(str)
OUT = "/content/drive/MyDrive/GSE178265_DA_neurons.h5ad"
da.write_h5ad(OUT, compression="gzip")
adata.obs.to_csv("/content/drive/MyDrive/GSE178265_all_cells_clusters.csv")
!ls -lh /content/drive/MyDrive/GSE178265_DA_neurons.h5ad
print(f"Saved {da.n_obs:,} dopamine neurons x {da.n_vars:,} genes")

## Cell 8b — Save labelled reference of all cell types (added 23 Oct)

In [ ]:
# CELL 8b — Save a labelled reference of ALL cell types, for Aim 1 (3–5 minutes)
adata.obs["cell_type"] = adata.obs["leiden"].astype(str).map(summary["best_guess"].to_dict())
adata.obs.loc[adata.obs["leiden"].astype(str).isin(da_clusters).to_numpy(), "cell_type"] = "DA_neuron"
adata.X = adata.layers["counts"]          # raw counts, in place (no extra memory)
del adata.layers["counts"]
for k in ["X_pca", "X_pca_harmony"]:
    adata.obsm.pop(k, None)
adata.write_h5ad("/content/drive/MyDrive/GSE178265_celltype_reference.h5ad", compression="gzip")
summary.to_csv("/content/drive/MyDrive/GSE178265_cluster_identities.csv")
!ls -lh /content/drive/MyDrive/GSE178265_celltype_reference.h5ad
print(adata.obs["cell_type"].value_counts())